# NBA Data Preprocessing

In [114]:
import pandas as pd
import unicodedata
import re
import os

In [2]:
players_raw = pd.read_csv(r'data\player_bios.csv', encoding='utf-8-sig')
mvp_raw = pd.read_csv(r'data\mvp_voting.csv', encoding='utf-8-sig')
player_total_stats_raw = pd.read_csv(r'data\player_total_stats.csv', encoding='utf-8-sig')
teams_raw = pd.read_csv(r"data\teams_info.csv", encoding="utf-8-sig")

In [50]:
def normalize_name(name):
    if pd.isna(name):
        return None
    return unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode("ascii").strip()


In [4]:
players_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1172 entries, 0 to 1171
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   player_id         1172 non-null   object 
 1   name              1172 non-null   object 
 2   position          1172 non-null   object 
 3   shoots            1172 non-null   object 
 4   height_cm         1172 non-null   int64  
 5   weight_kg         1172 non-null   int64  
 6   college           806 non-null    object 
 7   draft_year        749 non-null    float64
 8   nba_debut         1172 non-null   object 
 9   experience_years  1172 non-null   int64  
 10  is_active         1172 non-null   bool   
 11  birth_date        1172 non-null   object 
 12  birth_place       1172 non-null   object 
dtypes: bool(1), float64(1), int64(3), object(8)
memory usage: 111.1+ KB


In [5]:
players_raw.head()

,player_id,name,position,shoots,height_cm,weight_kg,college,draft_year,nba_debut,experience_years,is_active,birth_date,birth_place
0,hardeja01,James Harden,Point Guard and Shooting Guard,Left,196,99,Arizona State,2009.0,"October 28, 2009",17,True,1989-08-26,California
1,lillada01,Damian Lillard,Point Guard,Right,188,90,Weber State,2012.0,"October 31, 2012",13,True,1990-07-15,California
2,bookede01,Devin Booker,Shooting Guard and Point Guard,Right,196,93,Kentucky,2015.0,"October 28, 2015",11,True,1996-10-30,Michigan
3,antetgi01,Giannis Antetokounmpo,"Power Forward, Small Forward, Point Guard, and...",Right,211,110,NaN,2013.0,"October 30, 2013",13,True,1994-12-06,Greece
4,youngtr01,Trae Young,Point Guard,Right,188,74,Oklahoma,2018.0,"October 17, 2018",8,True,1998-09-19,Texas


In [6]:
mvp_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 22 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Rank       81 non-null     object 
 1   Player     81 non-null     object 
 2   Age        81 non-null     int64  
 3   Team       81 non-null     object 
 4   First      81 non-null     int64  
 5   Pts Won    81 non-null     int64  
 6   Pts Max    81 non-null     int64  
 7   Share      81 non-null     float64
 8   G          81 non-null     int64  
 9   MP         81 non-null     float64
 10  PTS        81 non-null     float64
 11  TRB        81 non-null     float64
 12  AST        81 non-null     float64
 13  STL        81 non-null     float64
 14  BLK        81 non-null     float64
 15  FG%        81 non-null     float64
 16  3P%        81 non-null     float64
 17  FT%        81 non-null     float64
 18  WS         81 non-null     float64
 19  WS/48      81 non-null     float64
 20  player_id  8

In [7]:
mvp_raw.head()

,Rank,Player,Age,Team,First,Pts Won,Pts Max,Share,G,MP,...,AST,STL,BLK,FG%,3P%,FT%,WS,WS/48,player_id,season
0,1,Giannis Antetokounmpo,25,MIL,85,962,1010,0.952,63,30.4,...,5.6,1.0,1.0,0.553,0.304,0.633,11.1,0.279,antetgi01,2019-20
1,2,LeBron James,35,LAL,16,753,1010,0.746,67,34.6,...,10.2,1.2,0.5,0.493,0.348,0.693,9.8,0.204,jamesle01,2019-20
2,3,James Harden,30,HOU,0,367,1010,0.363,68,36.5,...,7.5,1.8,0.9,0.444,0.355,0.865,13.1,0.254,hardeja01,2019-20
3,4,Luka DonÄiÄ,20,DAL,0,200,1010,0.198,61,33.6,...,8.8,1.0,0.2,0.463,0.316,0.758,8.8,0.207,doncilu01,2019-20
4,5,Kawhi Leonard,28,LAC,0,168,1010,0.166,57,32.4,...,4.9,1.8,0.6,0.470,0.378,0.886,8.7,0.226,leonaka01,2019-20


In [8]:
player_total_stats_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5057 entries, 0 to 5056
Data columns (total 34 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Rk             5050 non-null   float64
 1   Player         5057 non-null   object 
 2   Age            5050 non-null   float64
 3   Team           5050 non-null   object 
 4   Pos            5050 non-null   object 
 5   G              5050 non-null   float64
 6   GS             5050 non-null   float64
 7   MP             5050 non-null   float64
 8   FG             5050 non-null   float64
 9   FGA            5050 non-null   float64
 10  FG%            5020 non-null   float64
 11  3P             5050 non-null   float64
 12  3PA            5050 non-null   float64
 13  3P%            4749 non-null   float64
 14  2P             5050 non-null   float64
 15  2PA            5050 non-null   float64
 16  2P%            4979 non-null   float64
 17  eFG%           5020 non-null   float64
 18  FT      

In [9]:
player_total_stats_raw.head()

,Rk,Player,Age,Team,Pos,G,GS,MP,FG,FGA,...,AST,STL,BLK,TOV,PF,PTS,Trp-Dbl,Awards,player_url_id,season
0,1.0,James Harden,30.0,HOU,SG,68.0,68.0,2483.0,672.0,1514.0,...,512.0,125.0,60.0,308.0,227.0,2335.0,4.0,"MVP-3,AS,NBA1",hardeja01,2019-20
1,2.0,Damian Lillard,29.0,POR,PG,66.0,66.0,2474.0,624.0,1349.0,...,530.0,70.0,22.0,194.0,114.0,1978.0,1.0,"MVP-8,AS,NBA2",lillada01,2019-20
2,3.0,Devin Booker,23.0,PHO,SG,70.0,70.0,2512.0,627.0,1283.0,...,456.0,49.0,18.0,264.0,213.0,1863.0,0.0,AS,bookede01,2019-20
3,4.0,Giannis Antetokounmpo,25.0,MIL,PF,63.0,63.0,1917.0,685.0,1238.0,...,354.0,61.0,66.0,230.0,195.0,1857.0,4.0,"MVP-1,DPOY-1,AS,NBA1,DEF1",antetgi01,2019-20
4,5.0,Trae Young,21.0,ATL,PG,60.0,60.0,2120.0,546.0,1249.0,...,560.0,65.0,8.0,289.0,104.0,1778.0,2.0,AS,youngtr01,2019-20


In [10]:
teams_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   team_name                 30 non-null     object 
 1   league                    30 non-null     object 
 2   from                      30 non-null     object 
 3   to                        30 non-null     object 
 4   years                     30 non-null     int64  
 5   games                     30 non-null     int64  
 6   wins                      30 non-null     int64  
 7   losses                    30 non-null     int64  
 8   win_percentage            30 non-null     float64
 9   playoffs                  30 non-null     int64  
 10  division_championships    30 non-null     int64  
 11  conference_championships  30 non-null     int64  
 12  championships             30 non-null     int64  
dtypes: float64(1), int64(8), object(4)
memory usage: 3.2+ KB


In [11]:
teams_raw.head()

,team_name,league,from,to,years,games,wins,losses,win_percentage,playoffs,division_championships,conference_championships,championships
0,Atlanta Hawks,NBA,1949-50,2025-26,77,6102,3013,3088,0.494,50,13,0,1
1,Boston Celtics,NBA/BAA,1946-47,2025-26,80,6278,3751,2527,0.597,63,36,11,18
2,Brooklyn Nets,NBA/ABA,1967-68,2025-26,59,4776,2074,2702,0.434,31,5,2,2
3,Charlotte Hornets,NBA,1988-89,2025-26,36,2877,1237,1640,0.430,10,0,0,0
4,Chicago Bulls,NBA,1966-67,2025-26,60,4844,2453,2391,0.506,36,9,6,6


## 1. Players

In [121]:
players = players_raw.dropna(subset=["player_id"])
players = players.drop_duplicates(subset=["player_id"])

players = players.rename(columns={"player_id": "bref_id"})

players = players.sort_values("bref_id")
players = players.reset_index(drop=True)

players["player_id"] = players.index + 1001

players["name_normalized"] = players["name"].apply(normalize_name)

players["height_cm"] = pd.to_numeric(players["height_cm"], errors="coerce")
players["height_cm"] = players["height_cm"].astype("Int64")

players["weight_kg"] = pd.to_numeric(players["weight_kg"], errors="coerce")
players["weight_kg"] = players["weight_kg"].astype("Int64")

players["draft_year"] = pd.to_numeric(players["draft_year"], errors="coerce")
players["draft_year"] = players["draft_year"].astype("Int64")

players["experience_years"] = pd.to_numeric(players["experience_years"], errors="coerce")
players["experience_years"] = players["experience_years"].astype("Int64")

players["nba_debut"] = pd.to_datetime(players["nba_debut"], errors="coerce")
players["nba_debut"] = players["nba_debut"].dt.year
players["nba_debut"] = players["nba_debut"].astype("Int64")

players["birth_date"] = pd.to_datetime(players["birth_date"], errors="coerce")

players["is_active"] = players["is_active"].astype("boolean")

id_map = dict(zip(players["bref_id"], players["player_id"]))

players = players[[
    "player_id",
    "name_normalized",
    "position",
    "shoots",
    "height_cm",
    "weight_kg",
    "draft_year",
    "nba_debut",
    "experience_years",
    "is_active",
    "birth_date",
    "birth_place",
]]

players = players.rename(columns={"name_normalized": "name"})


In [122]:
players.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1172 entries, 0 to 1171
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   player_id         1172 non-null   int64         
 1   name              1172 non-null   object        
 2   position          1172 non-null   object        
 3   shoots            1172 non-null   object        
 4   height_cm         1172 non-null   Int64         
 5   weight_kg         1172 non-null   Int64         
 6   draft_year        749 non-null    Int64         
 7   nba_debut         1172 non-null   Int64         
 8   experience_years  1172 non-null   Int64         
 9   is_active         1172 non-null   boolean       
 10  birth_date        1172 non-null   datetime64[ns]
 11  birth_place       1172 non-null   object        
dtypes: Int64(5), boolean(1), datetime64[ns](1), int64(1), object(4)
memory usage: 108.9+ KB


In [123]:
players.head()

,player_id,name,position,shoots,height_cm,weight_kg,draft_year,nba_debut,experience_years,is_active,birth_date,birth_place
0,1001,Precious Achiuwa,Center and Power Forward,Right,203,110,2020,2020,6,True,1999-09-19,Nigeria
1,1002,Jaylen Adams,Point Guard,Right,183,102,<NA>,2018,3,False,1996-05-04,Maryland
2,1003,Steven Adams,Center,Right,211,120,2013,2013,12,True,1993-07-20,New Zealand
3,1004,Bam Adebayo,Center and Power Forward,Right,206,115,2017,2017,9,True,1997-07-18,New Jersey
4,1005,Ochai Agbaji,Shooting Guard,Right,196,97,2022,2022,4,True,2000-04-20,Wisconsin


## 2. Teams

In [100]:
teams = teams_raw.dropna(subset=["team_name"])
teams = teams.drop_duplicates(subset=["team_name"])

teams = teams.sort_values("team_name").reset_index(drop=True)

teams["team_id"] = teams.index + 1

teams = teams.rename(columns={
    "from": "from_year",
    "to": "to_year",
})

teams["from_year"] = pd.to_numeric(teams["from_year"].astype(str).str[:4], errors="coerce").astype("Int64")
teams["to_year"]   = pd.to_numeric(teams["to_year"].astype(str).str[:4],   errors="coerce").astype("Int64")

teams["years"] = pd.to_numeric(teams["years"], errors="coerce").astype("Int64")


team_id_map = dict(zip(teams["team_name"], teams["team_id"]))

teams = teams[[
    "team_id", "team_name", "league", "from_year", "to_year", "years"
]]


In [101]:
teams.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   team_id    30 non-null     int64 
 1   team_name  30 non-null     object
 2   league     30 non-null     object
 3   from_year  30 non-null     Int64 
 4   to_year    30 non-null     Int64 
 5   years      30 non-null     Int64 
dtypes: Int64(3), int64(1), object(2)
memory usage: 1.6+ KB


In [102]:
teams.head()

,team_id,team_name,league,from_year,to_year,years
0,1,Atlanta Hawks,NBA,1949,2025,77
1,2,Boston Celtics,NBA/BAA,1946,2025,80
2,3,Brooklyn Nets,NBA/ABA,1967,2025,59
3,4,Charlotte Hornets,NBA,1988,2025,36
4,5,Chicago Bulls,NBA,1966,2025,60


## 3. Season

In [124]:
all_seasons = player_total_stats_raw["season"].unique()

season_df = pd.DataFrame({"season": sorted(all_seasons)})
season_df["season_id"] = season_df.index + 1

In [125]:
season_id_map = dict(zip(season_df["season"], season_df["season_id"]))

season_df = season_df[["season_id", "season"]]
season_df

,season_id,season
0,1,2019-20
1,2,2020-21
2,3,2021-22
3,4,2022-23
4,5,2023-24
5,6,2024-25
6,7,2025-26


## 4. Player_Season_Stats

In [105]:
abbr_to_team_name = {
    "ATL": ["Atlanta Hawks"], "BOS": ["Boston Celtics"],
    "BRK": ["Brooklyn Nets"], "BKN": ["Brooklyn Nets"],
    "CHO": ["Charlotte Hornets"], "CHA": ["Charlotte Hornets"],
    "CHI": ["Chicago Bulls"], "CLE": ["Cleveland Cavaliers"],
    "DAL": ["Dallas Mavericks"], "DEN": ["Denver Nuggets"],
    "DET": ["Detroit Pistons"], "GSW": ["Golden State Warriors"],
    "HOU": ["Houston Rockets"], "IND": ["Indiana Pacers"],
    "LAC": ["LA Clippers", "Los Angeles Clippers"], "LAL": ["Los Angeles Lakers"],
    "MEM": ["Memphis Grizzlies"], "MIA": ["Miami Heat"],
    "MIL": ["Milwaukee Bucks"], "MIN": ["Minnesota Timberwolves"],
    "NOP": ["New Orleans Pelicans"], "NYK": ["New York Knicks"],
    "OKC": ["Oklahoma City Thunder"], "ORL": ["Orlando Magic"],
    "PHI": ["Philadelphia 76ers"], "PHO": ["Phoenix Suns"], "PHX": ["Phoenix Suns"],
    "POR": ["Portland Trail Blazers"], "SAC": ["Sacramento Kings"],
    "SAS": ["San Antonio Spurs"], "TOR": ["Toronto Raptors"],
    "UTA": ["Utah Jazz"], "WAS": ["Washington Wizards"],
}

# codes bref uses for a player's combined multi-team row in a season "2TM"/"3TM"/"4TM"/"5TM"
MULTI_TEAM_CODES = re.compile(r"^(\d+TM)$")

def map_team_abbr_to_id(abbr):
    for name in abbr_to_team_name.get(abbr, []):
        if name in team_id_map:
            return team_id_map[name]
    return pd.NA

In [106]:
pss = player_total_stats_raw.copy()

pss = pss.dropna(subset=["player_url_id", "Team"]).copy()

# bref adds one combined "multi-team" row per player per season when they
# played for more than one team (labelled "2TM"/"3TM"/"4TM"/"5TM")
# on top of the individual per-team rows.
# We keep the per-team rows only, since Player_Season_Stats is keyed by
# (player_id, team_id, season_id) and these codes are not real teams.
pss = pss[~pss["Team"].astype(str).str.match(MULTI_TEAM_CODES)].copy()

pss["player_id"] = pss["player_url_id"].map(id_map)
pss["team_id"]   = pss["Team"].apply(map_team_abbr_to_id)
pss["season_id"] = pss["season"].map(season_id_map)

pss = pss.dropna(subset=["player_id", "team_id", "season_id"])

pss = pss.rename(columns={
    "Rk": "rank", "Age": "age", "Pos": "position",
    "G": "games", "GS": "games_started", "MP": "minutes_played",
    "FG": "field_goals", "FGA": "field_goals_attempted", "FG%": "field_goal_percentage",
    "3P": "three_point_field_goals", "3PA": "three_point_field_goals_attempted",
    "3P%": "three_point_field_goal_percentage",
    "2P": "two_point_field_goals", "2PA": "two_point_field_goals_attempted",
    "2P%": "two_point_field_goal_percentage",
    "eFG%": "effective_field_goal_percentage",
    "FT": "free_throws", "FTA": "free_throws_attempted", "FT%": "free_throw_percentage",
    "ORB": "offensive_rebounds", "DRB": "defensive_rebounds", "TRB": "total_rebounds",
    "AST": "assists", "STL": "steals", "BLK": "blocks",
    "TOV": "turnovers", "PF": "personal_fouls", "PTS": "points",
    "Trp-Dbl": "triple_doubles",
})

# INT columns
pss_int_cols = [
    "rank", "age", "games", "games_started", "minutes_played",
    "field_goals", "field_goals_attempted",
    "three_point_field_goals", "three_point_field_goals_attempted",
    "two_point_field_goals", "two_point_field_goals_attempted",
    "free_throws", "free_throws_attempted",
    "offensive_rebounds", "defensive_rebounds", "total_rebounds",
    "assists", "steals", "blocks", "turnovers", "personal_fouls",
    "points", "triple_doubles",
]
for col in pss_int_cols:
    pss[col] = pd.to_numeric(pss[col], errors="coerce").astype("Int64")

# DECIMAL(5,3) columns
pss_pct_cols = [
    "field_goal_percentage", "three_point_field_goal_percentage",
    "two_point_field_goal_percentage", "effective_field_goal_percentage",
    "free_throw_percentage",
]
for col in pss_pct_cols:
    pss[col] = pd.to_numeric(pss[col], errors="coerce").round(3)

pss["player_id"] = pss["player_id"].astype("Int64")
pss["team_id"]   = pss["team_id"].astype("Int64")
pss["season_id"] = pss["season_id"].astype("Int64")

# safety net for the composite primary key
pss = pss.drop_duplicates(subset=["player_id", "team_id", "season_id"])

player_season_stats = pss[[
    "player_id", "team_id", "season_id", "rank", "age", "position",
    "games", "games_started", "minutes_played",
    "field_goals", "field_goals_attempted", "field_goal_percentage",
    "three_point_field_goals", "three_point_field_goals_attempted", "three_point_field_goal_percentage",
    "two_point_field_goals", "two_point_field_goals_attempted", "two_point_field_goal_percentage",
    "effective_field_goal_percentage",
    "free_throws", "free_throws_attempted", "free_throw_percentage",
    "offensive_rebounds", "defensive_rebounds", "total_rebounds",
    "assists", "steals", "blocks", "turnovers", "personal_fouls",
    "points", "triple_doubles",
]].reset_index(drop=True)

In [107]:
player_season_stats.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4513 entries, 0 to 4512
Data columns (total 32 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   player_id                          4513 non-null   Int64  
 1   team_id                            4513 non-null   Int64  
 2   season_id                          4513 non-null   Int64  
 3   rank                               4513 non-null   Int64  
 4   age                                4513 non-null   Int64  
 5   position                           4513 non-null   object 
 6   games                              4513 non-null   Int64  
 7   games_started                      4513 non-null   Int64  
 8   minutes_played                     4513 non-null   Int64  
 9   field_goals                        4513 non-null   Int64  
 10  field_goals_attempted              4513 non-null   Int64  
 11  field_goal_percentage              4476 non-null   float

In [108]:
player_season_stats.head()

,player_id,team_id,season_id,rank,age,position,games,games_started,minutes_played,field_goals,...,offensive_rebounds,defensive_rebounds,total_rebounds,assists,steals,blocks,turnovers,personal_fouls,points,triple_doubles
0,1415,11,1,1,30,SG,68,68,2483,672,...,70,376,446,512,125,60,308,227,2335,4
1,1623,25,1,2,29,PG,66,66,2474,624,...,33,251,284,530,70,22,194,114,1978,1
2,1101,24,1,3,23,SG,70,70,2512,627,...,29,268,297,456,49,18,264,213,1863,0
3,1022,17,1,4,25,PF,63,63,1917,685,...,140,716,856,354,61,66,230,195,1857,4
4,2166,1,1,5,21,PG,60,60,2120,546,...,32,223,255,560,65,8,289,104,1778,2


In [109]:
player_season_stats.tail()

,player_id,team_id,season_id,rank,age,position,games,games_started,minutes_played,field_goals,...,offensive_rebounds,defensive_rebounds,total_rebounds,assists,steals,blocks,turnovers,personal_fouls,points,triple_doubles
4508,2017,13,7,578,24,PF,8,0,32,0,...,0,3,3,1,1,0,1,5,1,0
4509,1131,6,7,579,26,SG,1,0,3,0,...,0,1,1,0,0,0,0,0,0,0
4510,1308,5,7,580,19,PF,2,0,6,0,...,0,0,0,0,1,0,0,1,0,0
4511,1312,20,7,581,24,SF,5,0,8,0,...,1,1,2,0,0,0,0,1,0,0
4512,1442,28,7,582,22,PG,2,0,13,0,...,0,1,1,2,1,0,1,2,0,0


## 5. MVP

In [110]:
mvp_raw.head()

,Rank,Player,Age,Team,First,Pts Won,Pts Max,Share,G,MP,...,AST,STL,BLK,FG%,3P%,FT%,WS,WS/48,player_id,season
0,1,Giannis Antetokounmpo,25,MIL,85,962,1010,0.952,63,30.4,...,5.6,1.0,1.0,0.553,0.304,0.633,11.1,0.279,antetgi01,2019-20
1,2,LeBron James,35,LAL,16,753,1010,0.746,67,34.6,...,10.2,1.2,0.5,0.493,0.348,0.693,9.8,0.204,jamesle01,2019-20
2,3,James Harden,30,HOU,0,367,1010,0.363,68,36.5,...,7.5,1.8,0.9,0.444,0.355,0.865,13.1,0.254,hardeja01,2019-20
3,4,Luka DonÄiÄ,20,DAL,0,200,1010,0.198,61,33.6,...,8.8,1.0,0.2,0.463,0.316,0.758,8.8,0.207,doncilu01,2019-20
4,5,Kawhi Leonard,28,LAC,0,168,1010,0.166,57,32.4,...,4.9,1.8,0.6,0.470,0.378,0.886,8.7,0.226,leonaka01,2019-20


In [130]:
mvp = mvp_raw.copy()

# Rank can contain ties like "4T" -> keep only the numeric part
mvp["rank"] = mvp["Rank"].astype(str).str.extract(r"(\d+)").astype("Int64")

mvp["player_id"] = mvp["player_id"].map(id_map)
mvp["season_id"] = mvp["season"].map(season_id_map)
mvp["team_id"]   = mvp["Team"].apply(map_team_abbr_to_id)

mvp = mvp.dropna(subset=["player_id", "season_id"])

# games / MP / PTS / TRB / AST / STL / BLK / FG% / 3P% / FT% are dropped on
# purpose: that season's box-score numbers are already in Player_Season_Stats,
# so MVP_Votes only keeps what's specific to the MVP vote itself.
mvp = mvp.rename(columns={
    "First": "first_place_votes", "Pts Won": "points_won", "Pts Max": "points_max",
    "Share": "share", "WS": "win_shares", "WS/48": "win_shares_per_48",
})

# INT columns
mvp_int_cols = ["first_place_votes", "points_won", "points_max"]
for col in mvp_int_cols:
    mvp[col] = pd.to_numeric(mvp[col], errors="coerce").astype("Int64")

# DECIMAL(5,3) column
mvp["share"] = pd.to_numeric(mvp["share"], errors="coerce").round(3)

# DECIMAL(5,1) columns
mvp_1dec_cols = ["win_shares", "win_shares_per_48"]
for col in mvp_1dec_cols:
    mvp[col] = pd.to_numeric(mvp[col], errors="coerce").round(3)

mvp["player_id"] = mvp["player_id"].astype("Int64")
mvp["season_id"] = mvp["season_id"].astype("Int64")

mvp_votes = mvp[[
    "player_id", "season_id", "team_id", "rank", "first_place_votes", "points_won", "points_max",
    "share", "win_shares", "win_shares_per_48",
]].reset_index(drop=True)

In [131]:
mvp_votes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   player_id          81 non-null     Int64  
 1   season_id          81 non-null     Int64  
 2   team_id            79 non-null     object 
 3   rank               81 non-null     Int64  
 4   first_place_votes  81 non-null     Int64  
 5   points_won         81 non-null     Int64  
 6   points_max         81 non-null     Int64  
 7   share              81 non-null     float64
 8   win_shares         81 non-null     float64
 9   win_shares_per_48  81 non-null     float64
dtypes: Int64(6), float64(3), object(1)
memory usage: 6.9+ KB


In [132]:
mvp_votes.head()

,player_id,season_id,team_id,rank,first_place_votes,points_won,points_max,share,win_shares,win_shares_per_48
0,1022,1,17,1,85,962,1010,0.952,11.1,0.279
1,1512,1,14,2,16,753,1010,0.746,9.8,0.204
2,1415,1,11,3,0,367,1010,0.363,13.1,0.254
3,1271,1,7,4,0,200,1010,0.198,8.8,0.207
4,1615,1,13,5,0,168,1010,0.166,8.7,0.226


## Export Data

In [133]:
out_dir = "processed_data"
os.makedirs(out_dir, exist_ok=True)

exports = {
    # "players.csv": players,
    # "teams.csv": teams,
    # "season.csv": season_df,
    # "player_season_stats.csv": player_season_stats,
    "mvp_votes.csv": mvp_votes,
}

for fname, df in exports.items():
    path = os.path.join(out_dir, fname)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"wrote {path}  ({len(df)} rows, {len(df.columns)} cols)")

wrote processed_data\mvp_votes.csv  (81 rows, 10 cols)
